# 01 — Data Audit

Phase 2 of the Cassie project. Goal: understand what's actually in the
Olist tables before assuming anything about churn definitions, feature
availability, or data quality thresholds.

Run this after downloading the CSVs into `data/raw/` (see
`docs/data_download_instructions.md`). Every finding here feeds the
values that were left as "TBD, needs real data" in
`docs/data_quality_plan.md` and `docs/implementation_plan.md`.


In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

RAW = Path("../data/raw")

FILES = {
    "customers": "olist_customers_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

missing = [name for name, fname in FILES.items() if not (RAW / fname).exists()]
if missing:
    print("Missing files — download these before continuing:", missing)
else:
    print("All expected files found in", RAW.resolve())


All expected files found in C:\Users\USER\Desktop\ML PORTFOLIO\cassie\data\raw


## 1. Load everything, basic shape

In [3]:
dfs = {name: pd.read_csv(RAW / fname) for name, fname in FILES.items() if (RAW / fname).exists()}

shape_summary = pd.DataFrame(
    {name: [df.shape[0], df.shape[1]] for name, df in dfs.items()},
    index=["rows", "columns"],
).T
shape_summary


,rows,columns
customers,99441,5
orders,99441,8
order_items,112650,7
payments,103886,5
reviews,99224,7
products,32951,9
sellers,3095,4
geolocation,1000163,5
category_translation,71,2


## 2. Null rates per column

Anything above ~0 here needs a decision recorded in
`docs/data_quality_plan.md` — drop, impute, or keep-with-caveat, and
why.

In [4]:
for name, df in dfs.items():
    null_pct = (df.isnull().mean() * 100).round(2)
    null_pct = null_pct[null_pct > 0].sort_values(ascending=False)
    print(f"--- {name} ---")
    print(null_pct if len(null_pct) else "no nulls")
    print()


--- customers ---
no nulls

--- orders ---
order_delivered_customer_date    2.98
order_delivered_carrier_date     1.79
order_approved_at                0.16
dtype: float64

--- order_items ---
no nulls

--- payments ---
no nulls

--- reviews ---
review_comment_title      88.34
review_comment_message    58.70
dtype: float64

--- products ---
product_category_name         1.85
product_name_lenght           1.85
product_description_lenght    1.85
product_photos_qty            1.85
product_weight_g              0.01
product_length_cm             0.01
product_height_cm             0.01
product_width_cm              0.01
dtype: float64

--- sellers ---
no nulls

--- geolocation ---
no nulls

--- category_translation ---
no nulls



## 3. Date ranges and impossible timestamps

Checks the specific critical rule from `docs/data_quality_plan.md`:
delivery dates before purchase dates, and the overall date span of the
dataset (needed to pick a sensible temporal train/validation/test
cutoff in Phase 6/8).

In [5]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
orders = dfs["orders"].copy()
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")

print("Date range (purchase timestamp):", orders["order_purchase_timestamp"].min(), "to", orders["order_purchase_timestamp"].max())
print()

impossible = orders[
    orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"]
]
print(f"Orders delivered before purchase (should be 0): {len(impossible)}")

future_orders = orders[orders["order_purchase_timestamp"] > pd.Timestamp.now()]
print(f"Orders with a future purchase timestamp (should be 0): {len(future_orders)}")


Date range (purchase timestamp): 2016-09-04 21:15:19 to 2018-10-17 17:30:18

Orders delivered before purchase (should be 0): 0
Orders with a future purchase timestamp (should be 0): 0


## 4. Delivery duration distribution

Needed to set the "impossible delivery duration" threshold left open
in the data-quality plan.

In [6]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.days

print(orders["delivery_days"].describe())
print()
print("Negative delivery_days (impossible):", (orders["delivery_days"] < 0).sum())
print("Top 1% cutoff (candidate outlier threshold):", orders["delivery_days"].quantile(0.99))


count    96476.000000
mean        12.094086
std          9.551746
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
Name: delivery_days, dtype: float64

Negative delivery_days (impossible): 0
Top 1% cutoff (candidate outlier threshold): 46.0


## 5. Negative / invalid financial values

In [7]:
items = dfs["order_items"]
payments = dfs["payments"]

print("Negative price:", (items["price"] < 0).sum())
print("Negative freight_value:", (items["freight_value"] < 0).sum())
print("Negative payment_value:", (payments["payment_value"] < 0).sum())
print("payment_value == 0 (check if legitimate, e.g. vouchers):", (payments["payment_value"] == 0).sum())


Negative price: 0
Negative freight_value: 0
Negative payment_value: 0
payment_value == 0 (check if legitimate, e.g. vouchers): 9


## 6. Orphan foreign keys

In [8]:
order_ids = set(orders["order_id"])

for name in ["order_items", "payments", "reviews"]:
    df = dfs[name]
    orphans = (~df["order_id"].isin(order_ids)).sum()
    print(f"{name}: {orphans} rows with order_id not present in orders")


order_items: 0 rows with order_id not present in orders
payments: 0 rows with order_id not present in orders
reviews: 0 rows with order_id not present in orders


## 7. Duplicate primary keys

In [9]:
checks = {
    "orders": ("order_id", dfs["orders"]),
    "products": ("product_id", dfs["products"]),
    "sellers": ("seller_id", dfs["sellers"]),
}
for name, (key, df) in checks.items():
    dupes = df[key].duplicated().sum()
    print(f"{name}: {dupes} duplicate {key} values")


orders: 0 duplicate order_id values
products: 0 duplicate product_id values
sellers: 0 duplicate seller_id values


## 8. `customer_id` vs `customer_unique_id` — the identity check

This is the single most important check in the whole audit. If
`customer_id` and `customer_unique_id` turned out to be effectively
1:1, the "persistent identity" framing in the data dictionary would be
wrong and the churn definition would need rethinking. Confirming the
real ratio here, not assuming it.

In [10]:
customers = dfs["customers"]

n_customer_id = customers["customer_id"].nunique()
n_unique_id = customers["customer_unique_id"].nunique()

print(f"Distinct customer_id:        {n_customer_id}")
print(f"Distinct customer_unique_id: {n_unique_id}")
print(f"Ratio (customer_id / unique): {n_customer_id / n_unique_id:.3f}")
print()

orders_per_customer = customers.groupby("customer_unique_id")["customer_id"].nunique()
print("Orders (customer_id rows) per customer_unique_id — distribution:")
print(orders_per_customer.describe())
print()
print(f"customer_unique_ids with >1 customer_id (i.e. repeat buyers): "
      f"{(orders_per_customer > 1).sum()} "
      f"({(orders_per_customer > 1).mean() * 100:.2f}% of all customers)")


Distinct customer_id:        99441
Distinct customer_unique_id: 96096
Ratio (customer_id / unique): 1.035

Orders (customer_id rows) per customer_unique_id — distribution:
count    96096.000000
mean         1.034809
std          0.214384
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         17.000000
Name: customer_id, dtype: float64

customer_unique_ids with >1 customer_id (i.e. repeat buyers): 2997 (3.12% of all customers)


## 9. Repeat-purchase / inter-purchase gap behavior

Feeds directly into Phase 6 (churn definition) and the 90/120/180/270
day window comparison — need the actual gap distribution, not a
guess.

In [12]:
orders_with_customer = orders.merge(
    customers[["customer_id", "customer_unique_id"]], on="customer_id", how="left"
)

repeat_customers = orders_with_customer.groupby("customer_unique_id").filter(
    lambda g: len(g) > 1
)

gaps = (
    repeat_customers.sort_values("order_purchase_timestamp")
    .groupby("customer_unique_id")["order_purchase_timestamp"]
    .apply(lambda s: s.diff().dt.days.dropna())
)

print("Inter-purchase gap (days) distribution, repeat customers only:")
print(gaps.describe())
print()
for pct in [0.5, 0.75, 0.9, 0.95, 0.99]:
    print(f"  {int(pct*100)}th percentile: {gaps.quantile(pct):.0f} days")


Inter-purchase gap (days) distribution, repeat customers only:
count    3345.000000
mean       77.860389
std       107.410890
min         0.000000
25%         0.000000
50%        28.000000
75%       119.000000
max       608.000000
Name: order_purchase_timestamp, dtype: float64

  50th percentile: 28 days
  75th percentile: 119 days
  90th percentile: 241 days
  95th percentile: 312 days
  99th percentile: 448 days


## 10. Review score distribution

In [13]:
reviews = dfs["reviews"]
print(reviews["review_score"].value_counts(normalize=True).sort_index().round(3))
print()
print("Out-of-range scores (should be 0):",
      (~reviews["review_score"].between(1, 5)).sum())


review_score
1    0.115
2    0.032
3    0.082
4    0.193
5    0.578
Name: proportion, dtype: float64

Out-of-range scores (should be 0): 0


## 11. Category coverage against translation table

In [14]:
products = dfs["products"]
translation = dfs["category_translation"]

uncovered = set(products["product_category_name"].dropna()) - set(translation["product_category_name"])
print(f"Product categories with no English translation: {len(uncovered)}")
if uncovered:
    print(sorted(uncovered))


Product categories with no English translation: 2
['pc_gamer', 'portateis_cozinha_e_preparadores_de_alimentos']


In [15]:
 # Section 9b — is the 0-day gap real repeat purchase, or cart-splitting?
 
zero_gap_pairs = []
for cust_id, group in repeat_customers.sort_values("order_purchase_timestamp").groupby("customer_unique_id"):
    times = group["order_purchase_timestamp"].tolist()
    order_ids = group["order_id"].tolist()
    for i in range(1, len(times)):
        gap_days = (times[i] - times[i-1]).days
        if gap_days == 0:
            zero_gap_pairs.append((cust_id, order_ids[i-1], order_ids[i], times[i-1], times[i]))
 
print(f"Zero-day gap pairs found: {len(zero_gap_pairs)}")
print()
 
# Check: do these pairs share the exact same timestamp (strong cart-split signal)?
exact_same_ts = sum(1 for _, _, _, t1, t2 in zero_gap_pairs if t1 == t2)
print(f"Of those, exact same purchase_timestamp (to the second): {exact_same_ts} "
      f"({exact_same_ts / len(zero_gap_pairs) * 100:.1f}%)")
print()
 
# Sample a few to eyeball
sample = zero_gap_pairs[:5]
for cust_id, oid1, oid2, t1, t2 in sample:
    print(f"customer_unique_id={cust_id}")
    print(f"  order {oid1} at {t1}")
    print(f"  order {oid2} at {t2}")
print()
 
# How many DISTINCT purchase events does each repeat customer really have,
# if we collapse orders placed on the exact same timestamp into one event?
event_counts = orders_with_customer.groupby(
    ["customer_unique_id", "order_purchase_timestamp"]
).size().reset_index(name="orders_in_event")
 
true_events_per_customer = event_counts.groupby("customer_unique_id").size()
print("Distinct purchase EVENTS per customer_unique_id (collapsing same-timestamp orders):")
print(true_events_per_customer.describe())
print()
print(f"customer_unique_ids with >1 distinct purchase event: "
      f"{(true_events_per_customer > 1).sum()} "
      f"({(true_events_per_customer > 1).sum() / len(true_events_per_customer) * 100:.2f}% of all customers)")
 


Zero-day gap pairs found: 999

Of those, exact same purchase_timestamp (to the second): 292 (29.2%)

customer_unique_id=00cc12a6d8b578b8ebd21ea4e2ae8b27
  order 64307ceb91666760cf3ff463618302fd at 2017-03-21 19:25:22
  order d61b915b69851aec8a8865f36cfd793e at 2017-03-21 19:25:23
customer_unique_id=01a22e2079ea71e17313b88e5811e54a
  order 35d6f22df6139cb41e6dc813a0f84302 at 2018-01-22 22:45:49
  order 7f40591eeef659da2bc93d2735fa9476 at 2018-01-22 23:27:48
customer_unique_id=01ea7dfdac01a4e8fbe2902b73510b20
  order 4fafa25ee16300a26f7ef92f3d15b58b at 2018-02-09 17:36:42
  order 6cd2700e3e9fc015e3bbe4ee264af57b at 2018-02-09 17:36:45
customer_unique_id=027803eb28cc01fbdf5da72b109fabee
  order b1b74907a120888550f895aca299e24b at 2017-10-17 11:28:05
  order b2fc720676930e0f9f3c1e472e8cddad at 2017-10-17 11:28:18
customer_unique_id=02b20b7c813efede140142ac610e36dc
  order 25ce3a22fa2537206c22c8bef4f9b593 at 2018-03-19 20:50:39
  order 62ec63f37c8ed14ce3dba43a02721f5e at 2018-03-19 20:50:39

In [16]:
# Section 9c — rigorous churn-window decision: purchase events, exact coverage, IQR threshold
 
# --- Step 1: rebuild the gap distribution on PURCHASE EVENTS, not raw orders ---
# A purchase event = one (customer_unique_id, order_purchase_timestamp) pair.
# This removes cart-split inflation before we decide anything.
 
events = (
    orders_with_customer
    .drop_duplicates(subset=["customer_unique_id", "order_purchase_timestamp"])
    .sort_values(["customer_unique_id", "order_purchase_timestamp"])
)
 
event_counts_per_customer = events.groupby("customer_unique_id").size()
repeat_customer_ids = event_counts_per_customer[event_counts_per_customer > 1].index
 
repeat_events = events[events["customer_unique_id"].isin(repeat_customer_ids)]
 
event_gaps = (
    repeat_events
    .groupby("customer_unique_id")["order_purchase_timestamp"]
    .apply(lambda s: s.diff().dt.days.dropna())
)
 
print(f"Repeat customers (purchase-event basis): {len(repeat_customer_ids)} "
      f"({len(repeat_customer_ids) / event_counts_per_customer.shape[0] * 100:.2f}% of all customers)")
print(f"Total inter-purchase-event gaps: {len(event_gaps)}")
print()
print("Purchase-event gap distribution:")
print(event_gaps.describe())
print()
 
# --- Step 2: EXACT coverage at each candidate window, no interpolation ---
print("Exact % of gaps <= window (this is the real number, not an estimate):")
for window in [90, 120, 180, 270]:
    coverage = (event_gaps <= window).mean() * 100
    n_excluded = (event_gaps > window).sum()
    print(f"  {window:>3} days: {coverage:5.2f}% covered  |  "
          f"{n_excluded} genuine repeat-purchase gaps would be mislabeled as churn")
print()
 
# --- Step 3: cross-check cart-split classification against seller_id ---
# Does the split actually correspond to different sellers (the real mechanical
# cause), or same seller (genuinely two separate quick purchases)?
items_small = dfs["order_items"][["order_id", "seller_id"]].drop_duplicates()
 
split_seller_check = []
for cust_id, oid1, oid2, t1, t2 in zero_gap_pairs:
    sellers1 = set(items_small[items_small["order_id"] == oid1]["seller_id"])
    sellers2 = set(items_small[items_small["order_id"] == oid2]["seller_id"])
    different_sellers = len(sellers1 & sellers2) == 0
    split_seller_check.append(different_sellers)
 
pct_different_sellers = sum(split_seller_check) / len(split_seller_check) * 100
print(f"Of the {len(zero_gap_pairs)} zero-gap order pairs, "
      f"{pct_different_sellers:.1f}% involve completely different sellers "
      f"(supports cart-split-by-seller as the mechanism)")
print()
 
# --- Step 4: IQR-based delivery-duration outlier threshold (formal method) ---
q1 = orders["delivery_days"].quantile(0.25)
q3 = orders["delivery_days"].quantile(0.75)
iqr = q3 - q1
formal_threshold = q3 + 1.5 * iqr
 
print(f"Delivery days — Q1: {q1:.1f}, Q3: {q3:.1f}, IQR: {iqr:.1f}")
print(f"Formal outlier threshold (Q3 + 1.5*IQR): {formal_threshold:.1f} days")
print(f"Orders past this threshold: {(orders['delivery_days'] > formal_threshold).sum()} "
      f"({(orders['delivery_days'] > formal_threshold).mean() * 100:.2f}% of orders)")
 


Repeat customers (purchase-event basis): 2740 (2.85% of all customers)
Total inter-purchase-event gaps: 3053

Purchase-event gap distribution:
count    3053.000000
mean       85.307239
std       109.569304
min         0.000000
25%         2.000000
50%        38.000000
75%       132.000000
max       608.000000
Name: order_purchase_timestamp, dtype: float64

Exact % of gaps <= window (this is the real number, not an estimate):
   90 days: 66.72% covered  |  1016 genuine repeat-purchase gaps would be mislabeled as churn
  120 days: 73.11% covered  |  821 genuine repeat-purchase gaps would be mislabeled as churn
  180 days: 82.31% covered  |  540 genuine repeat-purchase gaps would be mislabeled as churn
  270 days: 91.48% covered  |  260 genuine repeat-purchase gaps would be mislabeled as churn

Of the 999 zero-gap order pairs, 62.4% involve completely different sellers (supports cart-split-by-seller as the mechanism)

Delivery days — Q1: 6.0, Q3: 15.0, IQR: 9.0
Formal outlier threshold (Q

## 12. Summary / findings to carry into `docs/`

Fill this in after running the cells above — this is the section that
gets promoted into `docs/12_data_quality_report.md` and used to finalize
the open thresholds in `docs/data_quality_plan.md` and the churn-window
comparison in Phase 6.

- Date range covered:
- Delivery-duration outlier threshold decided on:
- Rows dropped/flagged and why:
- `customer_id` : `customer_unique_id` ratio, and what it implies:
- Inter-purchase gap distribution — does it support 90/120/180/270 as
  reasonable churn-window candidates?
- Any other surprises worth documenting before Phase 3 (schema) locks
  in assumptions:
